<a href="https://colab.research.google.com/github/meijo-yoshikawa-Lab-student/2026labseminar/blob/colabtest/%E7%99%BA%E5%B1%95%E8%AA%B2%E9%A1%8C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## 6. 発展課題（穴埋め）：どのクラスを間違えやすい？

全体の精度だけでは「どの種類（クラス）でつまずいているか」は分かりません。
そこで、**クラスごとの正答率**と、**混同行列**（どのクラスをどのクラスと取り違えたかを表にしたもの）を作ります。

下のコードには **5か所のTODO** があります。各TODOには「何をするか」だけ書いてあるので、コードは自分で考えて埋めてください。

**ヒント**
- 出発点は、セクション2の穴埋め `task.py` の **TODO②**（予測を取り出して正解数を数える部分）です。発展課題はこれを「クラスごと」に数えるよう拡張します。
- **TODO③④**（クラスごとの正解数・総数を数える）は、`task.py` のTODO②を1クラスずつに分けたものと考えると分かりやすいです。
- **TODO①⑤**（混同行列）は、`confusion[正解のクラス][予測したクラス]` のように2次元の表のマス目を1つずつ増やしていきます。
- 調べるときのキーワード：「**Python リスト内包表記 2次元**」「**混同行列 confusion matrix とは**」「**PyTorch torch.max 予測ラベル**」


In [ ]:
# ======================================================================
#  アドバンスド課題（模範解答版・デバッグ用）: クラスごとの正答率 ＋ 混同行列
#  5箇所の TODO を埋めてください。かなり難しめです。
#  各 TODO には「何をするか」だけ書いてあります。コードは自分で考えること。
#  ヒント: 穴埋め task.py の TODO② が出発点。それを「クラス別」に拡張する。
# ======================================================================
import torch

def per_class_accuracy(net, testloader, device, num_classes=10):
    """クラスごとの正答率と混同行列を計算して返す。
    戻り値: (per_class_acc, confusion, overall_acc)
      - per_class_acc[k] : クラス k の正答率（float）
      - confusion[t][p]  : 正解が t で予測が p だった件数（int）
      - overall_acc      : 全体の正答率（float）
    """
    net.to(device); net.eval()
    correct = [0] * num_classes
    total = [0] * num_classes

    # ================================================================
    # TODO ①: 混同行列を初期化する
    #   num_classes 行 × num_classes 列 の、すべて 0 の二次元リストを作る。
    #   （confusion[t][p] に「正解t・予測p」の件数を入れていく）
    # ----------------------------------------------------------------
    _________________________________
    # ================================================================

    with torch.no_grad():
        for batch in testloader:
            images = batch["image"].to(device)
            labels = batch["label"].to(device)
            outputs = net(images)

            # ============================================================
            # TODO ②: 予測ラベルを取り出す
            #   モデル出力 outputs から、各サンプルの予測クラスを取り出す。
            # ------------------------------------------------------------
            ______________________________________
            # ============================================================

            for label, pred in zip(labels, predicted):
                t = label.item()
                p = pred.item()

                # ====================================================
                # TODO ③: 正解クラス t の総数を 1 増やす
                # ----------------------------------------------------
                __________________________________________
                # ====================================================

                # ====================================================
                # TODO ④（2行）: 予測が正解と一致していたら、
                #         正解クラス t の正解数を 1 増やす（if文 + カウントの2行）
                # ----------------------------------------------------
                ________________________________________________
                ________________________________________________
                # ====================================================

                # ====================================================
                # TODO ⑤: 混同行列の該当要素（正解t・予測p）を 1 増やす
                # ----------------------------------------------------
                _____________________________________________________
                # ====================================================

    per_class_acc = [correct[k] / total[k] if total[k] else 0.0 for k in range(num_classes)]
    overall_acc = sum(correct) / sum(total) if sum(total) else 0.0
    return per_class_acc, confusion, overall_acc

# ✅ 答え合わせ（TODO①〜⑤のチェック）
# 小さなダミーデータで per_class_accuracy を動かし、
# 手元で計算した正しい混同行列・正答率と一致するか確認します。
def _check_per_class():
    torch.manual_seed(2)
    _net = Net(); _net.eval()
    _imgs = torch.randn(40, 1, 28, 28); _lbls = torch.randint(0, 10, (40,))
    _loader = [{"image": _imgs, "label": _lbls}]
    _pca, _conf, _overall = per_class_accuracy(_net, _loader, torch.device("cpu"), num_classes=10)

    # 手元で正しい答えを計算
    with torch.no_grad():
        _pred = _net(_imgs).argmax(1)
    _conf_ref = [[0]*10 for _ in range(10)]
    _correct = [0]*10; _total = [0]*10
    for _t, _p in zip(_lbls.tolist(), _pred.tolist()):
        _conf_ref[_t][_p] += 1; _total[_t] += 1
        if _t == _p: _correct[_t] += 1
    _pca_ref = [_correct[k]/_total[k] if _total[k] else 0.0 for k in range(10)]
    _overall_ref = sum(_correct) / sum(_total)

    _ok_conf = (_conf == _conf_ref)
    _ok_pca = all(abs(a-b) < 1e-9 for a, b in zip(_pca, _pca_ref))
    _ok_overall = abs(_overall - _overall_ref) < 1e-9
    return _ok_conf, _ok_pca, _ok_overall

_oc, _op, _oo = _check_per_class()
print("TODO①⑤ 混同行列:", "✅ OK" if _oc else "❌ NG")
print("TODO②③④ クラス別正答率:", "✅ OK" if (_op and _oo) else "❌ NG")
if _oc and _op and _oo:
    print("\nすべて正しく書けています。下のセルで結果を見てみましょう。")
else:
    print("\nTODOの番号をもう一度確認してみてください。")



### 6-1. 結果を見る：クラス別の正答率と混同行列
埋めた関数を使って、クラスごとの正答率（棒グラフ）と混同行列（色の濃さで件数を表した図）を表示します。
混同行列の見方：縦が「本当の正解」、横が「AIの予測」。**対角線が濃いほど正しく当てられている**ことを表します。


In [ ]:
import torch
from torch.utils.data import DataLoader
tl, vl = load_data(0, 5, "mnist")
net = Net(); train(net, tl, 2, get_default_device())
per_class_acc, confusion, overall = per_class_accuracy(net, vl, get_default_device())
print("全体の正答率:", round(overall, 4))

import matplotlib.pyplot as plt
# クラス別正答率（棒グラフ）
plt.figure(figsize=(7,4)); plt.bar(range(10), per_class_acc)
plt.xticks(range(10)); plt.ylim(0,1); plt.xlabel("クラス"); plt.ylabel("正答率")
plt.title("クラスごとの正答率")
for i,a in enumerate(per_class_acc): plt.text(i, a+0.02, f"{a:.2f}", ha="center", fontsize=8)
plt.show()

# 混同行列（ヒートマップ）
import numpy as np
conf = np.array(confusion)
plt.figure(figsize=(6,5))
plt.imshow(conf, cmap="Blues")
plt.colorbar(label="件数")
plt.xlabel("予測クラス"); plt.ylabel("正解クラス")
plt.title("混同行列")
plt.xticks(range(10)); plt.yticks(range(10))
for t in range(10):
    for p in range(10):
        if conf[t][p] > 0:
            plt.text(p, t, conf[t][p], ha="center", va="center", fontsize=7,
                     color="white" if conf[t][p] > conf.max()/2 else "black")
plt.tight_layout(); plt.show()

# 実際に間違えた画像を10枚見てみる（正解と予測のラベル付き）
show_misclassified(net, vl, get_default_device(), dataset_name="mnist", n=10)